# Ollama: контракт API, поток и память контекста

**Вопрос.** Как отличить пустой ответ модели от неправильного поля клиента, вычислить скорости по ответу API и предсказать влияние `num_ctx` на KV-кэш? Ноутбук работает без сервера на зафиксированных ответах; последняя лаборатория главы повторяет проверки с настоящей Ollama.

In [ ]:
import json
import platform

print({'python': platform.python_version(), 'contract': 'ollama-api-v1'})
chat_response = {
    'model': 'booklm:q4',
    'message': {'role': 'assistant', 'content': 'Готово', 'thinking': ''},
    'done': True, 'done_reason': 'stop',
    'load_duration': 2_000_000_000,
    'prompt_eval_count': 100, 'prompt_eval_duration': 500_000_000,
    'eval_count': 20, 'eval_duration': 1_000_000_000,
    'total_duration': 3_600_000_000,
}
generate_response = {'model': 'booklm:q4', 'response': 'Готово', 'done': True}

## 1. Endpoint определяет путь к тексту

У `/api/chat` текст находится в `message.content`; у `/api/generate` — в `response`. Универсальный клиент обязан знать схему, а не искать любое строковое поле.

In [ ]:
def final_text(endpoint, payload):
    if endpoint == '/api/chat':
        return payload['message']['content']
    if endpoint == '/api/generate':
        return payload['response']
    raise ValueError('unsupported endpoint')

assert final_text('/api/chat', chat_response) == 'Готово'
assert final_text('/api/generate', generate_response) == 'Готово'
wrong = chat_response.get('response', '')
assert wrong == ''  # модель ответила; клиент прочитал не ту схему

## 2. Скорости prefill и decode

Длительности Ollama заданы в наносекундах. Время загрузки не смешивается со скоростью генерации.

In [ ]:
def tokens_per_second(count, duration_ns):
    if count < 0 or duration_ns <= 0:
        raise ValueError('counts and duration must be positive')
    return count / (duration_ns / 1_000_000_000)

prefill_rate = tokens_per_second(
    chat_response['prompt_eval_count'], chat_response['prompt_eval_duration'])
decode_rate = tokens_per_second(
    chat_response['eval_count'], chat_response['eval_duration'])
assert prefill_rate == 200.0
assert decode_rate == 20.0
{'prefill_tok_s': prefill_rate, 'decode_tok_s': decode_rate,
 'load_s': chat_response['load_duration'] / 1e9}

## 3. Поток NDJSON и рассуждение

Поток — несколько самостоятельных JSON-строк. Первая запись может содержать рассуждение, а окончательный текст появится позже.

In [ ]:
ndjson = '\n'.join([
    json.dumps({'message': {'thinking': 'Проверяю', 'content': ''}, 'done': False}),
    json.dumps({'message': {'thinking': '', 'content': 'Го'}, 'done': False}),
    json.dumps({'message': {'thinking': '', 'content': 'тово'}, 'done': False}),
    json.dumps({'message': {'thinking': '', 'content': ''}, 'done': True,
                'done_reason': 'stop', 'eval_count': 4, 'eval_duration': 200_000_000}),
])

records = [json.loads(line) for line in ndjson.splitlines() if line.strip()]
thinking = ''.join(r['message'].get('thinking', '') for r in records)
content = ''.join(r['message'].get('content', '') for r in records)
assert thinking == 'Проверяю'
assert content == 'Готово'
assert records[-1]['done'] is True

## 4. Пустой `content` не всегда означает пустое вычисление

Если предел новых токенов закончился во время рассуждения, нужно сохранить `thinking`, `done_reason` и счётчики.

In [ ]:
truncated = {
    'message': {'thinking': 'Длинное внутреннее вычисление', 'content': ''},
    'done': True, 'done_reason': 'length', 'eval_count': 16,
}

def diagnose_empty_chat(payload):
    if payload.get('message', {}).get('content'):
        return 'content-present'
    if payload.get('message', {}).get('thinking') and payload.get('done_reason') == 'length':
        return 'limit-reached-during-thinking'
    if not payload.get('done'):
        return 'stream-not-finished'
    return 'inspect-full-payload'

assert diagnose_empty_chat(truncated) == 'limit-reached-during-thinking'

## 5. `num_ctx` масштабирует KV-кэш

Для обычного GQA-кэша на один токен нужны ключ и значение каждого слоя и KV-головы. Вес модели в расчёте не меняется.

In [ ]:
def kv_bytes(layers, kv_heads, head_dim, tokens, bytes_per_value=2):
    return 2 * layers * kv_heads * head_dim * tokens * bytes_per_value

short = kv_bytes(layers=32, kv_heads=8, head_dim=128, tokens=4096)
long = kv_bytes(layers=32, kv_heads=8, head_dim=128, tokens=32768)
assert long == 8 * short
{'ctx_4096_GiB': short / 1024**3, 'ctx_32768_GiB': long / 1024**3}

## 6. Паспорт настоящего запуска

Поля ниже заполняются выводами `ollama --version`, `ollama show --modelfile`, `ollama ps` и полным JSON API. Пустое значение означает непроверенный контракт, а не `auto`.

In [ ]:
manifest = {
    'ollama_version': None,
    'model_name_and_tag': None,
    'artifact_digest': None,
    'modelfile': None,
    'num_ctx': 4096,
    'num_predict': 128,
    'think': False,
    'processor_from_ollama_ps': None,
    'cold_response': None,
    'warm_response': None,
}
required_observations = ['ollama_version', 'model_name_and_tag', 'artifact_digest',
                         'modelfile', 'processor_from_ollama_ps']
assert all(key in manifest for key in required_observations)
manifest

## 7. Поиск неисправностей по уровням

Путь запроса в Ollama проходит пять уровней: имя с меткой (`booklm:q8`), файл GGUF, `Modelfile`, исполнитель и сервер. Ищите поломку сверху вниз.

1. **Ошибка соединения.** Сервер не запущен (`ollama serve`) или клиент стучится не по тому адресу: по умолчанию `http://127.0.0.1:11434`, переменная `OLLAMA_HOST` его меняет.
2. **«Модель не найдена».** Почти всегда неполное имя: `booklm` и `booklm:q8` — разные имена, без метки подставляется `:latest`. Список имён — `ollama list`.
3. **Свой GGUF не загружается.** Запустите его прямо в llama.cpp (`llama-cli -m файл.gguf`): так видно, читается ли сам файл.
4. **Медленный ответ.** Разложите `total_duration` на `load_duration`, обработку входа и генерацию (раздел 2 этого ноутбука). Большая доля загрузки — холодный запрос или маленький `keep_alive`; `ollama ps` с долей `CPU` — кэш на `num_ctx × OLLAMA_NUM_PARALLEL` не влез в видеопамять (раздел 5).
5. **Пустой `content`.** Посмотрите `done_reason`, `eval_count` и `message.thinking` (раздел 4): предел кончился в рассуждении, вход занял почти весь `num_ctx` или клиент читает не то поле (раздел 1).
6. **Ответ Ollama разошёлся с llama.cpp на том же GGUF.** Сначала сверьте токены входа и шаблон (`ollama show --modelfile`), потом `num_ctx`, параметры выборки и `num_predict`, и только потом подозревайте ядра.

### Справка по Modelfile

| Строка | Что задаёт |
|---|---|
| `FROM` | основа: имя готовой модели или путь к GGUF |
| `PARAMETER` | параметры выполнения: `num_ctx`, `temperature`, `top_k`, `stop`, `num_predict` |
| `SYSTEM` | системное сообщение по умолчанию |
| `TEMPLATE` | разметка ролей на языке шаблонов Go |
| `ADAPTER` | адаптер LoRA; подходит только к той основе, на которой обучен |

`num_ctx` задаётся в трёх местах: по умолчанию для сервера, в `Modelfile` и в `options` запроса; запрос переопределяет остальные, поэтому записывайте значение, которое реально применилось. Команды: `ollama pull` скачивает модель, `show` печатает описание, `ps` — что загружено и где, `stop` выгружает исполнитель.


## Задания

1. Добавьте время получения каждой NDJSON-строки и различите первую запись, первое рассуждение и первый окончательный символ.
2. Создайте ответ с `done=false` и докажите, что клиент не публикует его как законченный.
3. Сравните `/api/generate` и `/api/chat` на одной инструкции и объясните роль шаблона.
4. Заполните паспорт для настоящей Ollama, повторите холодный и тёплый запросы.
5. Измените только `num_ctx`, заранее посчитайте KV-кэш и сравните размещение из `ollama ps`.
6. Импортируйте один GGUF и сравните жадный результат с зафиксированной версией llama.cpp.